This script is used to simulate the average no. of zealots to kill before obtaining a summoning eye

Rules can be found here: https://hypixelskyblock.minecraft.wiki/w/Special_Zealot

In [ ]:
# import section
import numpy as np

In [ ]:
# config section

# --- PLAYER SETUP ---
# zealot type
is_briuser: bool = True
# Multiple Hits to Kill	x1.111 or /0.9
need_multi_hit: bool = False 
# Enderman Pet Level, linearly increase from 0 to 100. At max level, increase chance by 25%
enderman_pet_level: int = 100
# Zealuck level: divide by (1-0.02x) for each level of zealuck from 0 to 5. At max level, divide by 0.9
zealuck_level: int = 5
# Tracking: look at your skyblock profile. I can't help you with that
tracking: float = 0.0


# --- SIMULATION SETUP ---
# Number of simulations to run
num_kills: int = 1_000_000

In [ ]:
# probability calculation section. 
# note that this is calculating for denominator, so multiplication and division are reversed
raw_denominator: float = 420 - 40*is_briuser # 420 for briuser, 380 for non-briuser
cold_streak_threshold: int = raw_denominator # for later use in cold streak calculation
raw_denominator *= 0.9 if need_multi_hit else 1.0
raw_denominator /= (1 + 0.0025 * enderman_pet_level)
raw_denominator *= (1 - 0.02 * zealuck_level)
raw_denominator /= (1 + tracking/100)
print(f"Raw denominator: {raw_denominator:.2f}, which is a {100/raw_denominator:.2f}% chance of dropping an eye.")
print(f"Cold streak threshold: {cold_streak_threshold} kills")


In [ ]:
# simulation section
def consider_cold_streak(current_kills: int)->float:
    "This function determines whether the current kill is qualified for a cold streak buff. Returns the new probability directly"
    if current_kills <= cold_streak_threshold:
        return raw_denominator
    elif current_kills < 1.5*cold_streak_threshold:
        return raw_denominator / 2
    elif current_kills < 2*cold_streak_threshold:
        return raw_denominator / 3
    else:
        return raw_denominator / 4

current_kills: int = 0
eyes_dropped: int = 0
num_killed_arr: list[int] = []
roll: float = np.random.uniform(0, raw_denominator)
actual_denominator: float = raw_denominator
for i in range(num_kills):
    current_kills += 1
    # simulate the kill
    actual_denominator = consider_cold_streak(current_kills)
    roll = np.random.uniform(0, actual_denominator)
    if i%(num_kills//10) == 0:
       print(f"Total kill: {i}, Current kill: {current_kills}, current roll: {roll:.2f}, current denominator: {actual_denominator:.2f}")
    if roll <= 1:
        eyes_dropped += 1
        # if eyes_dropped % 10000 == 0:
        #     print(f"{eyes_dropped}th eye dropped after {current_kills} kills, averaging {i/eyes_dropped:.2f} kills per eye.")
        num_killed_arr.append(current_kills)
        current_kills = 0

In [ ]:
# Tally area
print(f"Total eyes dropped: {eyes_dropped}, averaging **{num_kills/eyes_dropped:.2f}** kills per eye.")
print(f"5-number summary of kills per eye:\nMin: {np.percentile(num_killed_arr, 0):.0f}\n LQ: {np.percentile(num_killed_arr, 25):.0f}\nMed: {np.percentile(num_killed_arr, 50):.0f}\n UQ: {np.percentile(num_killed_arr, 75):.0f}\nMax: {np.percentile(num_killed_arr, 100):.0f}")
print(f"Standard Deviation of kills per eye: {np.std(num_killed_arr):.2f}")

In [ ]:
# Optional figure area
import matplotlib.pyplot as plt

# Raw Distribution of kills per eye
plt.figure(figsize=(8, 6))
plt.hist(num_killed_arr, bins=20, edgecolor='black')
plt.xlabel('Kills per Eye')
plt.ylabel('Frequency')
plt.title('Distribution of Kills per Eye')
